
Building a GAN-Based AI Text Detector

Last Updated: July 14th, 2025

Daily Challenge : Building a GAN-Based AI Text Detector


👩‍🏫 👩🏿‍🏫 What You’ll learn

    How to train a Generative Adversarial Network (GAN) for detecting AI-generated text.
    How to use a pre-trained BERT model for sequence classification.
    How to preprocess text data and tokenize it for deep learning models.
    How to evaluate model performance using AUC scores.
    How to fine-tune and optimize deep learning models.
    How to perform inference and generate predictions on test data.


🛠️ What you will create

    A GAN-based model that detects AI-generated text using embeddings from a BERT model.
    A training pipeline that leverages a discriminator and generator network.
    A model that improves based on AUC scores for stability in training.
    A final submission file with predictions on the test dataset.


Dataset

You can find the dataset for this exercise here: Dataset


Task

For today’s challenge, you are provided with the final code with parts to fill. When you see a “TODO” it means you need to write code. Complete all of them.


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import random
import string

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import BertConfig
from transformers.models.bert.modeling_bert import BertEncoder
from sklearn.metrics import roc_auc_score

#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

TRAIN_PATH = TODO
TEST_PATH = TODO
PROMPT_PATH = TODO

src_train = TODO
src_prompt = TODO

src_sub = TODO


# Model preparation

tokenizer_save_path = TODO
model_save_path = TODO

tokenizer = TODO
pretrained_model = TODO
embedding_model = TODO

"""# Parameter definition"""

train_batch_size = TODO
test_batch_size = TODO
lr = TODO
beta1 = TODO
nz = 100  # Dimensions of the latent vector
num_epochs = TODO
num_hidden_layers = TODO
train_ratio = TODO

"""# Data Preparation"""

class GANDAIGDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

all_num = TODO
train_num = TODO
test_num = TODO


train_set = TODO
test_set = pd.concat([
    TODO,
]).reset_index(drop=True)


train_dataset = TODO
test_dataset = TODO

train_loader = TODO
test_loader = TODO

"""# Generator definition"""

config = BertConfig(num_hidden_layers=num_hidden_layers)

class Generator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Linear(input_dim, 256 * 128)

        self.conv_net = nn.Sequential(
            TODO
        )
        self.bert_encoder = BertEncoder(config)


    def forward(self, x):
        TODO
        return x

"""# Discriminator definition"""

class SumBertPooler(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        sum_hidden = hidden_states.sum(dim=1)
        sum_mask = sum_hidden.sum(1).unsqueeze(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)

        mean_embeddings = sum_hidden / sum_mask
        return mean_embeddings


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert_encoder = BertEncoder(config)
        self.bert_encoder.layer = nn.ModuleList([
            layer for layer in pretrained_model.bert.encoder.layer[:6]
        ])
        self.pooler = SumBertPooler()
        self.classifier = torch.nn.Sequential(
            TODO

        )

    def forward(self, input):
        out = self.bert_encoder(input)
        out = self.pooler(out.last_hidden_state)
        out = self.classifier(out)
        return torch.sigmoid(out).view(-1)

"""# Training"""

# Commented out IPython magic to ensure Python compatibility.
def eval_auc(model):
    model.eval()

    predictions = []
    actuals = []
    with torch.no_grad():
        for batch in test_loader:
            encodings = TODO
            input_ids = TODO
            token_type_ids = TODO
            embeded = TODO
            embeded =TODO
            attention_mask = TODO
            label = batch[1].float().to(device)

            outputs = model(embeded)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(label.cpu().numpy())

    auc = TODO
    print("AUC:", auc)
    return auc

def get_model_info_dict(model, epoch, auc_score):
    current_device = next(model.parameters()).device
    model.to('cpu')

    model_info = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'auc_score': auc_score,
    }

    model.to(current_device)
    return model_info

def preparation_embedding(texts):
    encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    input_ids = encodings['input_ids']
    token_type_ids = encodings['token_type_ids']
    embeded = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids)
    return embeded

def GAN_step(optimizerG, optimizerD, netG, netD, real_data, label, epoch, i):
    netD.zero_grad()
    batch_size = real_data.size(0)

    output = netD(real_data)
    errD_real = criterion(output, label)
    errD_real.backward()
    D_x = output.mean().item()

    noise = torch.randn(batch_size, nz, device=device)
    fake_data = netG(noise).last_hidden_state
    label.fill_(1)
    output = netD(fake_data.detach())
    errD_fake = criterion(output, label)
    errD_fake.backward()
    D_G_z1 = output.mean().item()
    errD = errD_real + errD_fake
    optimizerD.step()

    netG.zero_grad()
    label.fill_(0)
    output = netD(fake_data)
    errG = criterion(output, label)
    errG.backward()
    D_G_z2 = output.mean().item()
    optimizerG.step()
    if i % 50 == 0:
        print('[%d/%d][%d/%d] Loss_D: %.4f Loss_G: %.4f D(x): %.4f D(G(z)): %.4f / %.4f'
#               % (epoch, num_epochs, i, len(train_loader), errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

    return optimizerG, optimizerD, netG, netD

netG = TODO
netD = TODO

criterion = TODO
optimizerD = TODO
optimizerG = TODO

model_infos = []
for epoch in range(num_epochs):
    for i, data in enumerate(train_loader, 0):
        with torch.no_grad():
            embeded = preparation_embedding(data[0])

        optimizerG, optimizerD, netG, netD = GAN_step(
            optimizerG=TODO,
            optimizerD=TODO,
            netG=netG,
            netD=netD,
            real_data=embeded.to(device),
            label=data[1].float().to(device),
            epoch=epoch, i=i)

    auc_score = TODO
    model_infos.append(get_model_info_dict(netD, epoch, auc_score))

print('Train complete！')

"""# Inference"""

max_auc_model_info = TODO
model = Discriminator()
model.load_state_dict(max_auc_model_info['model_state_dict'])
model.to(device)
model.eval()

class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __getitem__(self, idx):
        return self.texts[idx]

    def __len__(self):
        return len(self.texts)

sub_dataset = TODO

inference_loader = TODO

sub_predictions = []
with torch.no_grad():
    for batch in inference_loader:
        encodings = TODO
        input_ids = TODO
        token_type_ids = TODO
        embeded = TODO
        embeded = embeded.to(device)

        outputs = model(embeded)
        sub_predictions.extend(outputs.cpu().numpy())

sub_ans_df = TODO
print(sub_ans_df)


Instructions :

1. Download the Dataset

    Upload the Kaggle API key.
    Move the key to the correct directory and set permissions, you may accept the rules of the competitions in Rulesor in Participate.
    Download and unzip the dataset.
    or :
    Download manually from Kaggle

2. Load the Data

    Read the training and test datasets using pandas.
    Display basic statistics and structure of the dataset.

3. Prepare the Model

    Load the BERT tokenizer and pre-trained model for sequence classification : bert-base-uncased.
    Extract embeddings from the BERT model to use in the GAN framework.

4. Set Hyperparameters

    Define batch sizes, learning rates, latent vector dimensions, and training epochs.

5. Prepare the Data for Training

    Create a PyTorch dataset class for handling text data.
    Split the data into training and testing sets.
    Use DataLoader to load batches efficiently.

6. Define the Generator Model

    Build a neural network that generates text embeddings using ConvTranspose1D layers.
    Incorporate a BERT encoder in the generator.

7. Define the Discriminator Model

    Extract and modify layers from a pre-trained BERT model.
    Implement a pooling mechanism for text classification.
    Construct a classification head using fully connected layers.

8. Train the Model

    Implement a GAN training loop.
    Train the generator to produce embeddings that fool the discriminator.
    Train the discriminator to differentiate between real and generated embeddings.
    Evaluate the model using AUC scores to monitor training stability.

9. Perform Inference

    Load the best-performing discriminator model based on AUC scores.
    Process test data through the model to generate predictions.


Mets les CSV Kaggle dans DATA_DIR (ou utilise le bloc Kaggle indiqué en comments).

   train_essays.csv, test_essays.csv, train_prompts.csv (facultatif), sample_submission.csv

In [ ]:
import os, json, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

from transformers import BertTokenizer, BertForSequenceClassification, BertConfig
from transformers.models.bert.modeling_bert import BertEncoder

# reproducibility
def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(123)


# configuration

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

DATA_DIR = "./data_llm_detect"   # <- put your .csv files here (train_essays.csv, test_essays.csv, etc.)
SAVE_DIR = "./artifacts_gan_detector"
os.makedirs(SAVE_DIR, exist_ok=True)

# Optional Kaggle (commented on purpose to keep script portable)
# - Upload kaggle.json to the runtime then:
# !mkdir -p ~/.kaggle
# !cp /content/kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c llm-detect-ai-generated-text -p /content/data
# !unzip -o /content/data/llm-detect-ai-generated-text.zip -d /content/data

# paths
TRAIN_PATH  = os.path.join(DATA_DIR, "train_essays.csv")
TEST_PATH   = os.path.join(DATA_DIR, "test_essays.csv")
PROMPT_PATH = os.path.join(DATA_DIR, "train_prompts.csv")           # optional for EDA only
SUB_PATH    = os.path.join(DATA_DIR, "sample_submission.csv")

# training toggles
MODE = "auto"          # "auto" | "vanilla" | "one_class"
POS_MIN = 20           # threshold to switch to one-class when positives are too rare
MAX_SEQ_LEN = 128

# hyperparameters
BATCH_TRAIN = 32
BATCH_VAL   = 64
BATCH_TEST  = 64

LR_D  = 2e-4
LR_G  = 2e-4
BETA1 = 0.5
NZ    = 100                  # latent vector size

EPOCHS_SUP = 2               # short supervised warmup for D (if enough positives)
EPOCHS_GAN = 5               # adversarial training epochs (increase for better results)
GRAD_CLIP = 1.0

NUM_HIDDEN_LAYERS = 6        # number of BERT encoder layers reused in D
TRAIN_RATIO = 0.8            # train/val split

# files existence
for p in [TRAIN_PATH, TEST_PATH, SUB_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing file: {p}")


# load datasets

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
sub_df   = pd.read_csv(SUB_PATH)

# normalize column names
train_df.columns = [c.strip() for c in train_df.columns]
test_df.columns  = [c.strip() for c in test_df.columns]

assert "text" in train_df.columns, "Expected 'text' in train_essays.csv"
assert "generated" in train_df.columns, "Expected 'generated' in train_essays.csv"
assert "text" in test_df.columns and "id" in test_df.columns

label_dist = train_df["generated"].value_counts(normalize=True)
print("Label distribution (train):")
print(label_dist)

n_pos = int((train_df["generated"] == 1).sum())
n_neg = int((train_df["generated"] == 0).sum())
print(f"Total rows: {len(train_df)} | positives: {n_pos} | negatives: {n_neg}")


# tokenizer / BERT

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# we only reuse the encoder from this model (classification head is unused)
pretrained_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=1)
embedding_model = pretrained_model.bert.to(DEVICE).eval()


# helpers

@torch.no_grad()
def embed_texts(texts):
    """Return last_hidden_state embeddings (batch, seq_len, hidden)."""
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors="pt"
    ).to(DEVICE)
    out = embedding_model(
        input_ids=enc["input_ids"],
        token_type_ids=enc["token_type_ids"],
        attention_mask=enc["attention_mask"]
    )
    return out.last_hidden_state                 # (B, T, H)

def make_loader_text_label(df, batch_size, shuffle):
    dataset = list(zip(df["text"].tolist(), df["generated"].astype(float).tolist()))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)

def make_loader_text(df, batch_size):
    dataset = df["text"].tolist()
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=False)


# split (stratified)

y = train_df["generated"].values
X = train_df["text"].values
sss = StratifiedShuffleSplit(n_splits=1, test_size=1-TRAIN_RATIO, random_state=123)
train_idx, val_idx = next(sss.split(X, y))
train_split = train_df.iloc[train_idx].reset_index(drop=True)
val_split   = train_df.iloc[val_idx].reset_index(drop=True)

# mode decision
if MODE == "auto":
    RUN_ONE_CLASS = (n_pos < POS_MIN)
elif MODE == "one_class":
    RUN_ONE_CLASS = True
else:
    RUN_ONE_CLASS = False

print(f"Training mode: {'ONE-CLASS (anomaly)' if RUN_ONE_CLASS else 'VANILLA (with supervised warmup if possible)'}")


# models (G & D)

# BERT encoder config shared by Generator/Discriminator blocks
bert_cfg = BertConfig(
    hidden_size=768,
    num_hidden_layers=NUM_HIDDEN_LAYERS,
    num_attention_heads=12,
    intermediate_size=3072,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)

class Generator(nn.Module):
    """Map z -> sequence of hidden states (B, T, H) then refine with a BertEncoder."""
    def __init__(self, latent_dim=NZ, seq_len=MAX_SEQ_LEN, hidden_size=768, cfg=bert_cfg):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_size = hidden_size

        self.fc = nn.Linear(latent_dim, 256 * seq_len)
        self.deconv = nn.Sequential(
            nn.ConvTranspose1d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(512, hidden_size, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(inplace=True),
        )
        self.encoder = BertEncoder(cfg)

    def forward(self, z):
        b = z.size(0)
        x = self.fc(z)                          # (B, 256*T)
        x = x.view(b, 256, self.seq_len)        # (B, 256, T)
        x = self.deconv(x)                      # (B, H, T)
        x = x.permute(0, 2, 1).contiguous()     # (B, T, H)

        attn_mask = torch.ones(b, 1, 1, self.seq_len, device=x.device)
        out = self.encoder(hidden_states=x, attention_mask=attn_mask, return_dict=True)
        return out.last_hidden_state            # (B, T, H)

class SumBertPooler(nn.Module):
    """Simple sum/mean pooling over sequence length."""
    def forward(self, hidden_states):
        # hidden_states: (B, T, H)
        summed = hidden_states.sum(dim=1)                       # (B, H)
        denom = torch.clamp(summed.sum(1, keepdim=True), min=1e-9)
        return summed / denom                                   # (B, H)

class Discriminator(nn.Module):
    """Discriminator returns logits (not sigmoid)."""
    def __init__(self, cfg=bert_cfg, reuse_layers=NUM_HIDDEN_LAYERS):
        super().__init__()
        self.encoder = BertEncoder(cfg)
        # reuse first N layers from pretrained BERT
        self.encoder.layer = nn.ModuleList([
            layer for layer in pretrained_model.bert.encoder.layer[:reuse_layers]
        ])
        self.pool = SumBertPooler()
        self.head = nn.Sequential(
            nn.Linear(cfg.hidden_size, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )

    def forward(self, hidden_states):
        b, t, h = hidden_states.size()
        attn_mask = torch.ones(b, 1, 1, t, device=hidden_states.device)
        out = self.encoder(hidden_states=hidden_states, attention_mask=attn_mask, return_dict=True)
        pooled = self.pool(out.last_hidden_state)
        logits = self.head(pooled)
        return logits.view(-1)


# evaluation

@torch.no_grad()
def evaluate_sup(model_D, loader_val):
    """Evaluate supervised AUC/AUPRC on val loader (text, label)."""
    model_D.eval()
    preds, trues = [], []
    for texts, labels in loader_val:
        emb = embed_texts(texts)
        logits = model_D(emb)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds.extend(probs)
        trues.extend(np.array(labels, dtype=float))
    if len(set(trues)) < 2:
        # degenerate case when val has only one class
        return 0.5, sum(trues)/len(trues)
    auc = roc_auc_score(trues, preds)
    ap  = average_precision_score(trues, preds)
    return float(auc), float(ap)

@torch.no_grad()
def infer_probs(model_D, loader_test, invert=False):
    """Return probabilities for 'generated' class.
       If invert=True, we compute 1 - p_human (one-class mode)."""
    model_D.eval()
    out_probs = []
    for texts in loader_test:
        emb = embed_texts(texts)
        logits = model_D(emb)
        p_human = torch.sigmoid(logits).cpu().numpy()  # D is trained to output 'human' in one-class
        if invert:
            out_probs.extend(1.0 - p_human)
        else:
            out_probs.extend(p_human)                  # in vanilla supervised, you would output p_generated; see note below
    return np.asarray(out_probs, dtype=float)


# data loaders

train_loader_sup = make_loader_text_label(train_split, BATCH_TRAIN, shuffle=True)
val_loader_sup   = make_loader_text_label(val_split,   BATCH_VAL,   shuffle=False)
test_loader_inf  = make_loader_text(test_df, BATCH_TEST)


# build models & optimizers

netD = Discriminator().to(DEVICE)
netG = Generator().to(DEVICE)

# In supervised warmup, we want: label=1 for AI/generated, 0 for human.
# BCEWithLogitsLoss(pos_weight=...) helps but we cap it to avoid exploding gradients.
pos_tr = int((train_split["generated"] == 1).sum())
neg_tr = int((train_split["generated"] == 0).sum())
pos_weight_value = min(neg_tr / max(pos_tr, 1), 20.0)  # cap at 20.0
pos_weight = torch.tensor([pos_weight_value], device=DEVICE)

criterion_sup = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion_gan = nn.BCEWithLogitsLoss()

optD_sup = optim.Adam(netD.parameters(), lr=LR_D, betas=(BETA1, 0.999))
optD     = optim.Adam(netD.parameters(), lr=LR_D, betas=(BETA1, 0.999))
optG     = optim.Adam(netG.parameters(), lr=LR_G, betas=(BETA1, 0.999))


# supervised warm-up for D (only if not one-class and there are positives)

best_auc, best_state = -1.0, None
if (not RUN_ONE_CLASS) and (pos_tr > 0):
    print("==> Supervised warm-up for Discriminator ...")
    for ep in range(1, EPOCHS_SUP + 1):
        netD.train()
        losses = []
        for texts, labels in train_loader_sup:
            labels = torch.tensor(labels, dtype=torch.float32, device=DEVICE)
            emb = embed_texts(texts)
            logits = netD(emb)                         # logits for 'generated' class
            loss = criterion_sup(logits, labels)
            optD_sup.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(netD.parameters(), GRAD_CLIP)
            optD_sup.step()
            losses.append(loss.item())
        auc, ap = evaluate_sup(netD, val_loader_sup)
        print(f"[Sup] Epoch {ep}/{EPOCHS_SUP} - loss {np.mean(losses):.4f} | AUC {auc:.4f} | AUPRC {ap:.4f}")
        if auc > best_auc:
            best_auc = auc
            best_state = netD.state_dict()
    if best_state is not None:
        netD.load_state_dict(best_state)
        print(f"Loaded best supervised D (AUC={best_auc:.4f})")
else:
    print("==> Skipping supervised warm-up (one-class mode or no positives).")


# GAN training

def gan_step(real_batch, log_prefix):
    """One adversarial step. In one-class: 'real' are *human* embeddings (label=1 for real, 0 for fake)."""
    netD.train(); netG.train()
    bsz = real_batch.size(0)

    # Discriminator update
    optD.zero_grad()
    target_real = torch.ones(bsz, device=DEVICE)
    target_fake = torch.zeros(bsz, device=DEVICE)

    # Real
    logits_real = netD(real_batch)
    loss_real = criterion_gan(logits_real, target_real)

    # Fake
    z = torch.randn(bsz, NZ, device=DEVICE)
    fake_batch = netG(z)
    logits_fake = netD(fake_batch.detach())
    loss_fake = criterion_gan(logits_fake, target_fake)

    loss_D = loss_real + loss_fake
    loss_D.backward()
    nn.utils.clip_grad_norm_(netD.parameters(), GRAD_CLIP)
    optD.step()

    # Generator update
    optG.zero_grad()
    logits_fake_for_G = netD(fake_batch)           # try to fool D: want 'real'
    loss_G = criterion_gan(logits_fake_for_G, target_real)
    loss_G.backward()
    nn.utils.clip_grad_norm_(netG.parameters(), GRAD_CLIP)
    optG.step()

    with torch.no_grad():
        D_x = torch.sigmoid(logits_real).mean().item()
        D_G_z1 = torch.sigmoid(logits_fake).mean().item()
        D_G_z2 = torch.sigmoid(logits_fake_for_G).mean().item()

    return loss_D.item(), loss_G.item(), D_x, D_G_z1, D_G_z2

print("==> Adversarial training ...")
model_snapshots = []
for epoch in range(1, EPOCHS_GAN + 1):
    # one-class: we need only *human* (generated==0) texts for real_data
    if RUN_ONE_CLASS:
        human_texts = train_split.loc[train_split["generated"] == 0, "text"].tolist()
        # iterate by chunks
        net_losses = []
        for i in range(0, len(human_texts), BATCH_TRAIN):
            texts_chunk = human_texts[i:i+BATCH_TRAIN]
            if len(texts_chunk) == 0: break
            with torch.no_grad():
                real_emb = embed_texts(texts_chunk)       # (B, T, H)
            lossD, lossG, Dx, DGz1, DGz2 = gan_step(real_emb, log_prefix="OC")
            net_losses.append((lossD, lossG))
            if i % (BATCH_TRAIN*8) == 0:
                print(f"[OC-GAN][{epoch}/{EPOCHS_GAN}] step={i//BATCH_TRAIN} Loss_D={lossD:.4f} Loss_G={lossG:.4f} D(x)={Dx:.3f} D(Gz)={DGz1:.3f}/{DGz2:.3f}")
        # evaluate as anomaly detector: score = 1 - p_human
        auc, ap = evaluate_sup(netD, val_loader_sup) if train_split["generated"].nunique()>1 else (0.5, 0.0)
        # store snapshot
        model_snapshots.append({"epoch": epoch, "state_dict": netD.state_dict(), "auc": auc, "auprc": ap})
        print(f"[OC-GAN] Epoch {epoch}/{EPOCHS_GAN} — AUC {auc:.4f} | AUPRC {ap:.4f}")
    else:
        # vanilla adversarial regularization over the whole training (use *human* for real in the GAN step)
        human_texts = train_split.loc[train_split["generated"] == 0, "text"].tolist()
        net_losses = []
        for i in range(0, len(human_texts), BATCH_TRAIN):
            texts_chunk = human_texts[i:i+BATCH_TRAIN]
            if len(texts_chunk) == 0: break
            with torch.no_grad():
                real_emb = embed_texts(texts_chunk)
            lossD, lossG, Dx, DGz1, DGz2 = gan_step(real_emb, log_prefix="V")
            net_losses.append((lossD, lossG))
            if i % (BATCH_TRAIN*8) == 0:
                print(f"[GAN][{epoch}/{EPOCHS_GAN}] step={i//BATCH_TRAIN} Loss_D={lossD:.4f} Loss_G={lossG:.4f} D(x)={Dx:.3f} D(Gz)={DGz1:.3f}/{DGz2:.3f}")
        auc, ap = evaluate_sup(netD, val_loader_sup)
        model_snapshots.append({"epoch": epoch, "state_dict": netD.state_dict(), "auc": auc, "auprc": ap})
        print(f"[GAN] Epoch {epoch}/{EPOCHS_GAN} — AUC {auc:.4f} | AUPRC {ap:.4f}")

print("Training complete.")


# pick best snapshot

if len(model_snapshots) == 0:
    # edge case: if GAN loop skipped, just keep current D
    best_state = netD.state_dict()
    best_auc, best_ap = -1.0, -1.0
else:
    best_item = max(model_snapshots, key=lambda x: x["auc"])
    best_state = best_item["state_dict"]
    best_auc, best_ap = best_item["auc"], best_item["auprc"]
    print(f"Best epoch: {best_item['epoch']} | AUC {best_auc:.4f} | AUPRC {best_ap:.4f}")

torch.save(best_state, os.path.join(SAVE_DIR, "best_discriminator.pt"))


# inference

# In one-class mode: D is trained to output p_human. We return p_generated = 1 - p_human.
# In vanilla mode with supervised warm-up, our D's output was trained on 'generated' labels,
# so you could directly return sigmoid(logits) as p_generated. To keep a single code path
# and be robust if warm-up was skipped, we invert only in one-class mode:
invert_probs = True if RUN_ONE_CLASS else False

preds = infer_probs(netD, test_loader_inf, invert=invert_probs)
submission = pd.DataFrame({"id": test_df["id"], "generated": preds})
sub_path = os.path.join(SAVE_DIR, "submission.csv")
submission.to_csv(sub_path, index=False)
print("Saved submission to:", sub_path)


# save run config

run_cfg = {
    "mode": "one_class" if RUN_ONE_CLASS else "vanilla",
    "pos_min": POS_MIN,
    "seq_len": MAX_SEQ_LEN,
    "batch": {"train": BATCH_TRAIN, "val": BATCH_VAL, "test": BATCH_TEST},
    "optim": {"lr_D": LR_D, "lr_G": LR_G, "beta1": BETA1, "grad_clip": GRAD_CLIP},
    "epochs": {"sup": EPOCHS_SUP, "gan": EPOCHS_GAN},
    "reuse_bert_layers": NUM_HIDDEN_LAYERS,
    "device": str(DEVICE),
    "best_auc": best_auc,
    "best_auprc": best_ap
}
with open(os.path.join(SAVE_DIR, "run_config.json"), "w") as f:
    json.dump(run_cfg, f, indent=2)
print("Artifacts saved to:", SAVE_DIR)